# Data Collection Notebook
## Customer Review Analysis - Collecting Reviews from Reddit and E-commerce Sites

This notebook demonstrates web scraping techniques to collect customer reviews from multiple sources including Reddit and e-commerce websites.

**Objectives:**
- Set up Reddit API access using PRAW
- Scrape product reviews from e-commerce websites
- Handle both static and dynamic web content
- Store collected data in structured formats

## 1. Import Required Libraries

Import all necessary libraries for web scraping, API access, and data manipulation.

In [1]:
# Core Data Science Libraries
import pandas as pd
import numpy as np
from datetime import datetime
import json
import time
import re

# Reddit API
import praw
from praw.models import MoreComments

# Web Scraping
import requests
from bs4 import BeautifulSoup
import lxml

# Dynamic Content Handling
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

# Utilities
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")
print(f"Current timestamp: {datetime.now()}")

ModuleNotFoundError: No module named 'praw'

## 2. Setup Reddit API with PRAW

Configure PRAW (Python Reddit API Wrapper) credentials and authenticate with Reddit API.

**Note:** You need to create a Reddit app at https://www.reddit.com/prefs/apps to get your credentials.

In [ ]:
# Reddit API Configuration
# Replace these with your actual credentials or use environment variables
REDDIT_CLIENT_ID = "your_client_id_here"
REDDIT_CLIENT_SECRET = "your_client_secret_here"
REDDIT_USER_AGENT = "CustomerReviewScraper/1.0 by YourUsername"

# Initialize Reddit instance
try:
    reddit = praw.Reddit(
        client_id=REDDIT_CLIENT_ID,
        client_secret=REDDIT_CLIENT_SECRET,
        user_agent=REDDIT_USER_AGENT
    )
    
    # Test connection
    print(f"Successfully connected to Reddit API")
    print(f"Read-only mode: {reddit.read_only}")
    
except Exception as e:
    print(f"Error connecting to Reddit API: {e}")
    print("Please update your credentials in the cell above")

In [ ]:
def scrape_reddit_reviews(subreddit_name, search_query, limit=100):
    """
    Scrape product reviews from Reddit subreddit
    
    Parameters:
    -----------
    subreddit_name : str
        Name of the subreddit to search
    search_query : str
        Search term for finding relevant posts
    limit : int
        Maximum number of posts to retrieve
    
    Returns:
    --------
    list : List of dictionaries containing review data
    """
    reviews = []
    
    try:
        subreddit = reddit.subreddit(subreddit_name)
        
        # Search for relevant posts
        for submission in tqdm(subreddit.search(search_query, limit=limit), 
                               desc=f"Scraping r/{subreddit_name}"):
            
            # Extract post information
            post_data = {
                'source': 'reddit',
                'subreddit': subreddit_name,
                'post_id': submission.id,
                'title': submission.title,
                'text': submission.selftext,
                'author': str(submission.author),
                'score': submission.score,
                'upvote_ratio': submission.upvote_ratio,
                'num_comments': submission.num_comments,
                'created_utc': datetime.fromtimestamp(submission.created_utc),
                'url': submission.url,
                'permalink': submission.permalink
            }
            
            reviews.append(post_data)
            
            # Add delay to respect rate limits
            time.sleep(0.5)
    
    except Exception as e:
        print(f"Error scraping Reddit: {e}")
    
    return reviews

# Example usage (uncomment when credentials are set)
# reddit_reviews = scrape_reddit_reviews('ProductReviews', 'smartphone review', limit=50)
# print(f"Collected {len(reddit_reviews)} Reddit reviews")

## 3. Scrape Product Reviews from E-commerce Sites

Use requests and BeautifulSoup to scrape product reviews from e-commerce websites.

**Note:** This is a template that needs to be adapted for specific e-commerce sites.

In [ ]:
def scrape_ecommerce_page(url, headers=None):
    """
    Fetch HTML content from e-commerce page
    
    Parameters:
    -----------
    url : str
        URL of the product page
    headers : dict
        HTTP headers to include in request
    
    Returns:
    --------
    BeautifulSoup : Parsed HTML content
    """
    if headers is None:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept-Language': 'en-US,en;q=0.9',
            'Accept-Encoding': 'gzip, deflate, br',
            'Connection': 'keep-alive'
        }
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'lxml')
        return soup
    
    except requests.exceptions.RequestException as e:
        print(f"Error fetching page: {e}")
        return None

# Test with a sample URL
sample_url = "https://example-ecommerce.com/product-reviews"
print("Scraping function defined successfully")
print("Note: Update the URL with an actual e-commerce product page")

## 4. Extract Reviews Using BeautifulSoup

Parse HTML content and extract review text, ratings, dates, and user information.

This section provides template functions that need to be customized based on the target website's structure.

In [ ]:
def extract_reviews_generic(soup, selectors):
    """
    Extract reviews from parsed HTML using CSS selectors
    
    Parameters:
    -----------
    soup : BeautifulSoup
        Parsed HTML content
    selectors : dict
        Dictionary of CSS selectors for different review components
    
    Returns:
    --------
    list : List of extracted reviews
    """
    reviews = []
    
    if soup is None:
        return reviews
    
    try:
        # Find all review containers
        review_containers = soup.select(selectors.get('container', '.review'))
        
        for container in review_containers:
            review_data = {
                'source': 'ecommerce',
                'review_id': None,
                'title': None,
                'text': None,
                'rating': None,
                'author': None,
                'date': None,
                'verified_purchase': False,
                'helpful_count': 0
            }
            
            # Extract title
            title_elem = container.select_one(selectors.get('title', '.review-title'))
            if title_elem:
                review_data['title'] = title_elem.get_text(strip=True)
            
            # Extract text
            text_elem = container.select_one(selectors.get('text', '.review-text'))
            if text_elem:
                review_data['text'] = text_elem.get_text(strip=True)
            
            # Extract rating
            rating_elem = container.select_one(selectors.get('rating', '.rating'))
            if rating_elem:
                rating_text = rating_elem.get_text(strip=True)
                # Extract numeric rating (e.g., "4.5 out of 5 stars")
                rating_match = re.search(r'(\d+\.?\d*)', rating_text)
                if rating_match:
                    review_data['rating'] = float(rating_match.group(1))
            
            # Extract author
            author_elem = container.select_one(selectors.get('author', '.reviewer-name'))
            if author_elem:
                review_data['author'] = author_elem.get_text(strip=True)
            
            # Extract date
            date_elem = container.select_one(selectors.get('date', '.review-date'))
            if date_elem:
                review_data['date'] = date_elem.get_text(strip=True)
            
            # Check verified purchase
            verified_elem = container.select_one(selectors.get('verified', '.verified-purchase'))
            review_data['verified_purchase'] = verified_elem is not None
            
            # Extract helpful count
            helpful_elem = container.select_one(selectors.get('helpful', '.helpful-count'))
            if helpful_elem:
                helpful_text = helpful_elem.get_text(strip=True)
                helpful_match = re.search(r'(\d+)', helpful_text)
                if helpful_match:
                    review_data['helpful_count'] = int(helpful_match.group(1))
            
            reviews.append(review_data)
    
    except Exception as e:
        print(f"Error extracting reviews: {e}")
    
    return reviews

# Example selectors (customize for your target website)
example_selectors = {
    'container': 'div.review-item',
    'title': 'h3.review-title',
    'text': 'div.review-body',
    'rating': 'span.star-rating',
    'author': 'span.reviewer-name',
    'date': 'span.review-date',
    'verified': 'span.verified-badge',
    'helpful': 'span.helpful-votes'
}

print("Review extraction function defined")
print("Customize the selectors dictionary based on your target website's HTML structure")

In [ ]:
def scrape_multiple_pages(base_url, num_pages=5, selectors=None):
    """
    Scrape reviews from multiple pages
    
    Parameters:
    -----------
    base_url : str
        Base URL with page parameter placeholder (e.g., '{page}')
    num_pages : int
        Number of pages to scrape
    selectors : dict
        CSS selectors for review extraction
    
    Returns:
    --------
    list : Combined list of all reviews
    """
    all_reviews = []
    
    for page_num in tqdm(range(1, num_pages + 1), desc="Scraping pages"):
        try:
            # Format URL with page number
            url = base_url.format(page=page_num)
            
            # Fetch and parse page
            soup = scrape_ecommerce_page(url)
            
            if soup:
                # Extract reviews
                reviews = extract_reviews_generic(soup, selectors or example_selectors)
                all_reviews.extend(reviews)
                
                print(f"Page {page_num}: Extracted {len(reviews)} reviews")
            
            # Respectful delay between requests
            time.sleep(2)
        
        except Exception as e:
            print(f"Error on page {page_num}: {e}")
            continue
    
    return all_reviews

print("Multi-page scraping function defined")

## 5. Handle Dynamic Content with Selenium

Use Selenium WebDriver to scrape JavaScript-rendered content and paginated reviews.

This is essential for websites that load content dynamically using JavaScript.

In [ ]:
def setup_selenium_driver(headless=True):
    """
    Initialize Selenium WebDriver with Chrome
    
    Parameters:
    -----------
    headless : bool
        Run browser in headless mode (no GUI)
    
    Returns:
    --------
    webdriver : Selenium WebDriver instance
    """
    chrome_options = Options()
    
    if headless:
        chrome_options.add_argument('--headless')
    
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-blink-features=AutomationControlled')
    chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
    
    try:
        driver = webdriver.Chrome(options=chrome_options)
        print("Selenium WebDriver initialized successfully")
        return driver
    
    except Exception as e:
        print(f"Error initializing Selenium: {e}")
        print("Make sure ChromeDriver is installed and in your PATH")
        return None

# Initialize driver
# driver = setup_selenium_driver(headless=True)

In [ ]:
def scrape_dynamic_reviews(url, driver, scroll_times=3, wait_time=2):
    """
    Scrape reviews from dynamically loaded pages
    
    Parameters:
    -----------
    url : str
        URL of the product page
    driver : webdriver
        Selenium WebDriver instance
    scroll_times : int
        Number of times to scroll down (for infinite scroll)
    wait_time : int
        Seconds to wait after each scroll
    
    Returns:
    --------
    list : Extracted reviews
    """
    reviews = []
    
    if driver is None:
        print("WebDriver not initialized")
        return reviews
    
    try:
        # Navigate to URL
        driver.get(url)
        time.sleep(3)  # Wait for initial load
        
        # Handle cookie consent or popups if needed
        try:
            # Example: close cookie banner
            cookie_button = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "button.accept-cookies"))
            )
            cookie_button.click()
            time.sleep(1)
        except:
            pass  # No cookie banner or already closed
        
        # Scroll to load more content
        for i in range(scroll_times):
            # Scroll to bottom
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(wait_time)
            
            # Click "Load More" button if it exists
            try:
                load_more_button = driver.find_element(By.CSS_SELECTOR, "button.load-more-reviews")
                load_more_button.click()
                time.sleep(wait_time)
            except:
                pass  # No load more button
        
        # Get page source and parse with BeautifulSoup
        soup = BeautifulSoup(driver.page_source, 'lxml')
        
        # Extract reviews using the generic function
        reviews = extract_reviews_generic(soup, example_selectors)
        
        print(f"Extracted {len(reviews)} reviews from dynamic content")
    
    except Exception as e:
        print(f"Error scraping dynamic content: {e}")
    
    return reviews

print("Dynamic content scraping function defined")
print("Remember to close the driver when done: driver.quit()")

In [ ]:
def handle_pagination_selenium(base_url, driver, max_pages=5):
    """
    Handle pagination using Selenium by clicking next buttons
    
    Parameters:
    -----------
    base_url : str
        Starting URL
    driver : webdriver
        Selenium WebDriver instance
    max_pages : int
        Maximum number of pages to scrape
    
    Returns:
    --------
    list : All collected reviews
    """
    all_reviews = []
    
    if driver is None:
        print("WebDriver not initialized")
        return all_reviews
    
    try:
        driver.get(base_url)
        time.sleep(3)
        
        for page_num in tqdm(range(1, max_pages + 1), desc="Processing pages"):
            # Get current page content
            soup = BeautifulSoup(driver.page_source, 'lxml')
            reviews = extract_reviews_generic(soup, example_selectors)
            all_reviews.extend(reviews)
            
            print(f"Page {page_num}: Collected {len(reviews)} reviews")
            
            # Try to click next button
            try:
                next_button = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, "a.next-page, button.next"))
                )
                next_button.click()
                time.sleep(3)  # Wait for page to load
            except:
                print(f"No more pages found or next button not clickable")
                break
    
    except Exception as e:
        print(f"Error during pagination: {e}")
    
    return all_reviews

print("Pagination handler defined")

## 6. Save Raw Data to CSV/JSON

Export collected reviews to structured formats for further processing and analysis.

In [ ]:
def save_reviews_to_csv(reviews, filename='raw_reviews.csv'):
    """
    Save reviews to CSV file
    
    Parameters:
    -----------
    reviews : list
        List of review dictionaries
    filename : str
        Output filename
    """
    if not reviews:
        print("No reviews to save")
        return
    
    try:
        df = pd.DataFrame(reviews)
        
        # Add metadata
        df['collected_at'] = datetime.now()
        df['collection_date'] = datetime.now().date()
        
        # Save to CSV
        df.to_csv(filename, index=False, encoding='utf-8')
        
        print(f"✓ Saved {len(reviews)} reviews to {filename}")
        print(f"✓ File size: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")
        print(f"\nDataFrame shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        
        return df
    
    except Exception as e:
        print(f"Error saving to CSV: {e}")
        return None

def save_reviews_to_json(reviews, filename='raw_reviews.json'):
    """
    Save reviews to JSON file
    
    Parameters:
    -----------
    reviews : list
        List of review dictionaries
    filename : str
        Output filename
    """
    if not reviews:
        print("No reviews to save")
        return
    
    try:
        # Add metadata
        output_data = {
            'metadata': {
                'total_reviews': len(reviews),
                'collection_timestamp': datetime.now().isoformat(),
                'version': '1.0'
            },
            'reviews': reviews
        }
        
        # Save to JSON
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(output_data, f, indent=2, default=str, ensure_ascii=False)
        
        print(f"✓ Saved {len(reviews)} reviews to {filename}")
        
    except Exception as e:
        print(f"Error saving to JSON: {e}")

print("Data export functions defined")

In [ ]:
# Example: Combine all data sources and save

# Initialize empty list for all reviews
all_collected_reviews = []

# Add Reddit reviews (if collected)
# all_collected_reviews.extend(reddit_reviews)

# Add e-commerce reviews (if collected)
# ecommerce_reviews = scrape_multiple_pages(base_url, num_pages=5)
# all_collected_reviews.extend(ecommerce_reviews)

# Add dynamic content reviews (if collected)
# dynamic_reviews = scrape_dynamic_reviews(url, driver)
# all_collected_reviews.extend(dynamic_reviews)

# Save to both formats
# df_reviews = save_reviews_to_csv(all_collected_reviews, 'data/raw_reviews.csv')
# save_reviews_to_json(all_collected_reviews, 'data/raw_reviews.json')

print(f"Total reviews collected: {len(all_collected_reviews)}")
print("\nData collection pipeline ready!")
print("Uncomment the code above and update URLs/credentials to start collecting data")

## Summary & Next Steps

### What We've Built:
1. ✓ Reddit API integration using PRAW
2. ✓ Static web scraping with BeautifulSoup
3. ✓ Dynamic content handling with Selenium
4. ✓ Multi-page pagination support
5. ✓ Data export to CSV and JSON formats

### Before Running:
- Update Reddit API credentials
- Identify target e-commerce websites
- Customize CSS selectors for your target sites
- Install ChromeDriver for Selenium
- Create data directory: `mkdir -p data/`

### Next Notebook:
**02_data_preprocessing.ipynb** - Clean and preprocess the collected reviews

### Best Practices:
- Always respect robots.txt and terms of service
- Add delays between requests to avoid overwhelming servers
- Use appropriate user agents
- Handle errors gracefully
- Store data incrementally for large scraping jobs

In [ ]:
# Cleanup (run this cell when done scraping)
# if 'driver' in locals():
#     driver.quit()
#     print("✓ Selenium WebDriver closed")

print("Data Collection Notebook Complete!")